# LangChain Quickstart with Ollama

This notebook follows the official [LangChain Quickstart](https://docs.langchain.com/oss/python/langchain/quickstart) end to end, adapted to run **locally with Ollama** instead of a cloud API.

### What you will build

1. **Basic agent** — a weather tool + a single user question
2. **Research agent** — fetches *The Great Gatsby* from Project Gutenberg and answers structured questions
3. **LangChain vs Deep Agents** — compare `create_agent` with `create_deep_agent`

### Local model

We use **`qwen3:8b`** (must exist in your Ollama install). Change the model string if you prefer another tag from `ollama list`.

### Notebook map

| Section | Quickstart topic |
|---------|------------------|
| 1 | Install dependencies |
| 2 | Set up Ollama (+ common `ImportError` fix) |
| 3 | Build a basic agent |
| 4 | Build a real-world agent (system prompt, tools, model, memory) |
| 5 | Run LangChain agent vs deep agent |
| 6 | Trace with LangSmith (optional) |
| 7 | Next steps |

## 1. Install dependencies

The quickstart installs `langchain` and `deepagents`. For **Ollama**, you also need the provider integration package `langchain-ollama` — this is *not* pulled in automatically by `langchain` alone.

From the project root:

```bash
uv add langchain deepagents langchain-ollama
uv sync
```

Select the project kernel: **`.venv (Python 3.14.x)`** and restart it after installing packages.

In [1]:
# Verify required packages are importable in the active kernel.
# If any import fails, run `uv sync` from the project root and restart the kernel.

import langchain
import langchain_ollama
import deepagents

print(f"langchain         {langchain.__version__}")
print(f"langchain-ollama  {langchain_ollama.__version__}")
print(f"deepagents        {deepagents.__version__}")

langchain         1.3.11
langchain-ollama  1.1.0
deepagents        0.6.12


## 2. Set up Ollama

Per the [quickstart Ollama tab](https://docs.langchain.com/oss/python/langchain/quickstart):

- **Local:** Ollama must be running at [https://ollama.com](https://ollama.com) (default: `http://localhost:11434`)
- **Cloud:** set `OLLAMA_API_KEY` for hosted inference

For local development you typically **do not** need an API key.

---

### Troubleshooting: `ImportError` for `langchain-ollama`

If you copy the quickstart Ollama example before installing the integration package, `create_agent` fails when it tries to initialize the model:

```text
ImportError: Initializing ChatOllama requires the langchain-ollama package.
Please install it with `pip install langchain-ollama`
```

**Why:** LangChain uses a plugin model — each provider (OpenAI, Ollama, Gemini, …) lives in its own package. The string `"ollama:qwen3:8b"` tells LangChain to load `ChatOllama` from `langchain-ollama`.

**Fix:**

```bash
uv add langchain-ollama
# then restart the notebook kernel
```

Other common issues:

| Symptom | Fix |
|---------|-----|
| Connection refused | Start Ollama; check `http://localhost:11434` |
| Model not found | Run `ollama pull qwen3:8b` or match the tag from `ollama list` |
| Kernel spins forever | Restart kernel; select `.venv` interpreter |

In [2]:
# Confirm Ollama is reachable and the model is available.
# Uses the official ollama Python client (installed via langchain-ollama).

import ollama

OLLAMA_MODEL = "qwen3:8b"  # change if you use a different local model

models = [m.model for m in ollama.list().models]
print("Available models:", models)

if OLLAMA_MODEL not in models:
    raise RuntimeError(
        f"Model '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}"
    )

print(f"OK — '{OLLAMA_MODEL}' is ready.")

Available models: ['qwen3:8b', 'qwen3.5:9b', 'bge-m3:latest', 'mxbai-embed-large:latest', 'qwen3-vl:4b-instruct']
OK — 'qwen3:8b' is ready.


## 3. Build a basic agent

From the quickstart: create a simple agent with a language model, one tool, and a system prompt.

When you ask about the weather in San Francisco, the agent:

1. Reads the user message and available tools
2. Decides to call `get_weather("San Francisco")`
3. Composes a natural-language reply from the tool result

```text
User → LLM → tool call → get_weather → LLM → final answer
```

In [3]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a given city.

    Docstrings are exposed to the LLM as tool metadata — write them clearly.
    """
    return f"It's always sunny in {city}!"


# Model string format: "provider:model-name"
#   provider → routes to langchain-ollama (ChatOllama)
#   model    → must match an Ollama tag (see previous cell)
basic_agent = create_agent(
    model=f"ollama:{OLLAMA_MODEL}",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Agents accept a state dict with a "messages" list (chat API shape).
basic_result = basic_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)

# content_blocks holds structured output (text, tool calls, etc.)
print(basic_result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'The weather in San Francisco is currently sunny! 🌞 While it\'s often said that San Francisco is "always sunny," the forecast today aligns with that reputation. Let me know if you\'d like more details!'}]


## 4. Build a real-world agent

The quickstart research example answers questions about a text file fetched from the web. Along the way it covers:

1. **Detailed system prompts** — steer agent behavior
2. **Tools** — integrate with external data (`fetch_text_from_url`)
3. **Model configuration** — `init_chat_model` for `temperature`, `timeout`, etc.
4. **Conversational memory** — `InMemorySaver` checkpointer
5. **Deep Agents** — built-in planning, filesystem tools, subagents
6. **Testing** — run and compare outputs

> **Warning:** The Great Gatsby task sends a large document to the model and may use many tokens. Local 8B models may hit context limits or run slowly — that is expected.

### Step 1 — Define the system prompt

Keep it specific and actionable. Tell the agent what tools exist and what it must **not** guess.

In [3]:
SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

### Step 2 — Create tools

[Tools](https://docs.langchain.com/oss/python/langchain/tools) let the model call functions you define. The `@tool` decorator adds metadata (name, description, schema) that becomes part of the model prompt.

> Tool names, descriptions, and argument names should be well-documented — the LLM never sees your source code, only this metadata.

In [4]:
import urllib.error
import urllib.request

from langchain.tools import tool


@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    return raw.decode("utf-8", errors="replace")

### Step 3 — Configure your model

Use [`init_chat_model`](https://docs.langchain.com/oss/python/langchain/models) when you need explicit parameters. The quickstart Ollama example sets `temperature`, `timeout`, and `max_tokens`.

In [5]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    f"ollama:{OLLAMA_MODEL}",
    temperature=0.5,
    timeout=300,
    max_tokens=25000,
)

### Step 4 — Add memory

A [checkpointer](https://docs.langchain.com/oss/python/langgraph/add-memory#manage-short-term-memory) persists conversation state across turns. `InMemorySaver` is fine for notebooks; use a database-backed checkpointer in production.

In [6]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

### Step 5 — Create and run the agent

Two frameworks from the quickstart:

| API | When to use |
|-----|-------------|
| `create_agent` | Fine-grained control; you add capabilities yourself |
| `create_deep_agent` | Built-in planning, filesystem tools (`grep`, `read_file`), subagents |

Both agents share the same model, tools, system prompt, and checkpointer. Each run uses a distinct `thread_id` so memory does not leak between them.

In [7]:
from deepagents import create_deep_agent
from langchain.agents import create_agent

research_agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

deep_agent = create_deep_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

# Structured research task from the quickstart (Project Gutenberg — The Great Gatsby)
content = """Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

print("Running LangChain agent...")
agent_result = research_agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)
print(agent_result["messages"][-1].content_blocks)

print("\n" + "=" * 60 + "\n")
print("Running deep agent...")
deep_agent_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-da"}},
)
print(deep_agent_result["messages"][-1].content_blocks)

Running LangChain agent...
[{'type': 'text', 'text': '**Analysis of "The Great Gatsby" Chapter (Final Chapter):**\n\n---\n\n### **1. Key Themes and Symbolism**\n- **The Illusion of the American Dream**:  \n  Gatsby’s tragic pursuit of Daisy and his belief in the green light symbolize the corrupted American Dream. His dream is rooted in idealization, not reality, leading to his downfall. The green light, once a beacon of hope, becomes a metaphor for unattainable aspirations and the futility of chasing an idealized past.\n\n- **Moral Decay and Carelessness**:  \n  The novel critiques the moral emptiness of the wealthy elite (e.g., Tom and Daisy Buchanan). Their carelessness—symbolized by the "careless people" who "smash up things and creatures" and retreat into wealth—reflects the moral decay of the Jazz Age. Gatsby’s idealism contrasts with their selfishness, highlighting the clash between hope and corruption.\n\n- **The Past vs. the Present**:  \n  Gatsby’s inability to move on from hi

### Step 6 — Review the results

Outputs vary by model and run. The quickstart illustrates a typical pattern:

**LangChain agent (`create_agent`)** — often returns `null` for exact line counts because it only has `fetch_text_from_url`. Without `grep` or code execution it cannot reliably count lines in a large file.

**Deep agent (`create_deep_agent`)** — can plan with `write_todos`, save large tool output to the filesystem, and use built-in `grep` / `read_file` to produce verified counts (e.g. 258 lines with `Gatsby`, first `Daisy` at line 181 in the doc example).

With a local 8B model you may see shorter answers, timeouts, or context errors — compare the *behavior*, not only the numbers.

For LangChain agents, add more tools yourself to reach deep-agent capability; for maximum capability with minimal setup, prefer deep agents.

## 5. Trace agent calls (optional)

Complex agents make many LLM calls. [LangSmith](https://smith.langchain.com) logs traces so you can inspect each step.

Set environment variables before running agents:

```bash
export LANGSMITH_TRACING="true"
export LANGSMITH_API_KEY="..."
```

On Windows (PowerShell):

```powershell
$env:LANGSMITH_TRACING = "true"
$env:LANGSMITH_API_KEY = "your-key"
```

See the [tracing quickstart](https://docs.langchain.com/langsmith/trace-with-langchain).

In [ ]:
import os

# Optional: load LangSmith vars from a .env file at the project root.
# Uncomment if you use python-dotenv and have LANGSMITH_* in .env

# from pathlib import Path
# from dotenv import load_dotenv
# load_dotenv(Path("..") / ".env")

tracing_on = os.getenv("LANGSMITH_TRACING", "").lower() == "true"
has_key = bool(os.getenv("LANGSMITH_API_KEY"))

if tracing_on and has_key:
    print("LangSmith tracing is enabled — re-run an agent cell to capture traces.")
else:
    print("LangSmith not configured (optional). Set LANGSMITH_TRACING and LANGSMITH_API_KEY to enable.")

## 6. Next steps

You now have agents that can:

- **Understand context** and remember conversations (checkpointer + `thread_id`)
- **Use tools** intelligently
- **Provide structured responses** via `content_blocks`
- **Plan, research, and synthesize** (deep agents)

Continue with:

- **LangChain agents:** [memory](https://docs.langchain.com/oss/python/langgraph/add-memory#manage-short-term-memory), [deploy](https://docs.langchain.com/oss/python/langgraph/deploy)
- **Deep Agents:** [customization](https://docs.langchain.com/oss/python/deepagents/customization), [memory](https://docs.langchain.com/oss/python/deepagents/memory)
- **Other providers:** change the model string and install the matching package (`langchain-openai`, `langchain-ollama`, …)

---

*Source: [LangChain Quickstart](https://docs.langchain.com/oss/python/langchain/quickstart)*